# Day 2 — Orchestration: LangGraph State Graph (Vision Node → Analytical Node → HITL Gate)

**Scope of this notebook:** wiring the two frozen Day 1 nodes (Vision + Analytical) into a LangGraph
`StateGraph`, with a first-pass HITL conditional edge so the graph has a real terminal and is testable
end-to-end on your Day 1 canonical cases.

**Not yet built here (next session):**
- Groq-powered Pathologist Agent / Climate Agent (interpretive layer) — currently stubbed as pass-through placeholder nodes
- Translation Agent, gTTS audio, PDF report generation

**Mock mode:** This notebook runs in `MOCK_MODE = True` by default so the graph wiring can be tested
right now without your trained XGBoost artifact or GPU/ViT session loaded. The mock outputs are your
actual Day 1 canonical test results (Vision Test 1/2, Tabular Test 1/2/3), so the routing logic is being
validated against real numbers, not invented ones.

When you're back in your Day 1 Colab session (with `vision_pipeline/inference.py` and
`tabular_analytics/inference.py` importable and your trained XGBoost + SHAP explainer available), flip
`MOCK_MODE = False` and the same graph will call your real inference functions instead.


## 1. Frozen JSON contracts (from Day 1)

These mirror exactly what `day1_vision_node_results.md` and `day1_tabular_node_results.md` specify.
Nothing here should need to change when you swap mock → real inference — that's the point of freezing
the contract.


## 2. Shared graph state

One `TypedDict` that flows through every node. Each node reads what it needs and returns only the keys
it updates — LangGraph merges these into the running state.


## 3. Mock data — your actual Day 1 canonical results

Used only when `MOCK_MODE = True`, so the graph can be exercised end-to-end right now.


## 4. Node wrappers

Each wrapper tries real inference first when `MOCK_MODE = False`, otherwise falls back to the mock case
selected in state. This means the *same* graph and the *same* node functions work in both modes — you
don't rewrite the graph when you plug in the real models.


## 5. HITL gate + placeholder interpretive-agent nodes

Per `Remaining_tasks.txt`, the full intended order is:
`Vision Node → Analytical Node → Pathologist Agent + Climate Agent → HITL Gate`.

The Pathologist/Climate agents (Groq-backed interpretation of each JSON) aren't built yet — that's the
next task on your list. For now they're pass-through stubs so the graph shape is already correct and you
can drop the real Groq calls into `pathologist_agent_stub` / `climate_agent_stub` without touching the
graph wiring itself.

The HITL router only checks `vision_output.is_ambiguous`, matching the spec: a low-confidence disease
classification halts the pipeline regardless of what the tabular risk score says.


## 6. Build the graph


## 7. Test harness — canonical Day 1 cases run through the compiled graph

Three combinations, mirroring your actual Day 1 test data:

1. Confident vision (Test 1) + Moderate tabular (Test 1) → expect `proceed`
2. Ambiguous vision (Test 2) + High tabular (Test 3) → expect `ambiguous` (vision gate overrides tabular severity)
3. Confident vision (Test 1) + Low tabular (Test 2) → expect `proceed`


## 8. Next step

The graph shape is now correct and tested against real Day 1 numbers in mock mode. The next piece to
build (Day 2, item 2 on your task list) is replacing `pathologist_agent_stub` and `climate_agent_stub`
with real Groq API calls that read `vision_output` / `analytical_output` and produce plain-language
interpretations — those feed the Translation Agent once it exists.

Say the word when you want to move on to the Groq-powered interpretive agents.
